In [2]:
import os
import pandas as pd
import re
import numpy as np
import sys
from sklearn.feature_selection import f_classif
from scipy.stats import mode, wilcoxon
from statistics import mean

# ==================== CONFIGURACIÓN DE RUTAS ====================
datasets = ["braaksc", "ceradsc", "cogdx"]
k_values = [25, 50, 75, 100, 500, 1000, 1500, 2500]
results_root = "../results"
parsimony_summary_path = "../best_k_selection/parsimony_analysis/parsimony_summary.csv"

# Colores para reporte
GREEN, RED, YELLOW, BLUE = '\033[92m', '\033[91m', '\033[93m', '\033[94m'
BOLD, RESET = '\033[1m', '\033[0m'

# ==================== GESTIÓN DE SALIDA DE DATOS (LOGGER) ====================

class DualLogger(object):
    """
    Clase para redirigir la salida estándar simultáneamente a la consola 
    y a un archivo de texto de reporte.
    """
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# ==================== NÚCLEO ESTADÍSTICO ====================

def get_winner_info(dataset, k):
    py_path = os.path.join(results_root, dataset, f"k_{k}", "best_analysis", f"{dataset}_best_dataset_k{k}.py")
    if not os.path.exists(py_path): return None
    try:
        with open(py_path, 'r', encoding='utf-8') as f:
            content = f.read()
        match = re.search(r"metodologia_ganador\s*=\s*['\"](.+?)['\"]", content)
        return match.group(1) if match else None
    except: return None

def load_metric_vector(dataset, k, method, metric="BA"):
    base = f"genes-{dataset}"
    if "R2" in method: suffix = f"_{base}-FR-{k}.csv"
    elif "R3" in method: suffix = f"_{base}-resampling_FR-{k}.csv"
    else: suffix = f"_{base}_k{k}.csv"
    csv_path = os.path.join(results_root, dataset, f"k_{k}", f"test_{metric}{suffix}")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        col = method if method in df.columns else method.split("-")[0]
        if col in df.columns: return df[col].values[:10]
    return None

def calculate_win_loss_winner(dataset):
    scores_dict = {"BA": pd.DataFrame(), "F1": pd.DataFrame(), "PS": pd.DataFrame()}
    methods_found = {}
    for k in k_values:
        meth = get_winner_info(dataset, k)
        if not meth: continue
        methods_found[f"K={k}"] = meth
        for m in ["BA", "F1", "PS"]:
            vec = load_metric_vector(dataset, k, meth, m)
            if vec is not None: scores_dict[m][f"K={k}"] = vec
    if scores_dict["BA"].empty: return None, None

    def get_net_wins(df):
        labels = df.columns
        wins = {l: 0 for l in labels}
        for i in range(len(labels)):
            for j in range(i + 1, len(labels)):
                col_i, col_j = df.iloc[:, i].values, df.iloc[:, j].values
                if not np.array_equal(col_i, col_j):
                    try:
                        _, p = wilcoxon(col_i, col_j)
                        if p < 0.05:
                            if mean(col_i) > mean(col_j): 
                                wins[labels[i]] += 1; wins[labels[j]] -= 1
                            else: 
                                wins[labels[j]] += 1; wins[labels[i]] -= 1
                    except: pass
        return wins

    w_ba, w_f1, w_ps = get_net_wins(scores_dict["BA"]), get_net_wins(scores_dict["F1"]), get_net_wins(scores_dict["PS"])
    total_wins = {k: w_ba[k] + w_f1[k] + w_ps[k] for k in w_ba.keys()}
    best_k_label = max(total_wins, key=total_wins.get)
    return int(best_k_label.replace("K=", "")), methods_found[best_k_label]

def get_dataset_path(dataset, k, method):
    base = f"genes-{dataset}"
    if "R2" in method: filename = f"{base}-FR-{k}_train.csv"
    elif "R3" in method: filename = f"{base}-resampling_FR-{k}_train.csv"
    else: filename = f"{base}_train.csv"
    return os.path.join(results_root, dataset, f"k_{k}", filename)

# ==================== PROCESO PRINCIPAL ====================

os.makedirs("k_value_comparison", exist_ok=True)
log_report_path = "k_value_comparison/comparative_analysis_report.txt"
sys.stdout = DualLogger(log_report_path)

if not os.path.exists(parsimony_summary_path):
    print(f"{RED} No se encuentra resumen de parsimonia.{RESET}"); exit()

df_pars_global = pd.read_csv(parsimony_summary_path)

for dataset in datasets:
    print(f"\n{BOLD}{'='*100}{RESET}")
    print(f"{BOLD} COMPARATIVA DINÁMICA: {dataset.upper()}{RESET}")
    print(f"{BOLD}{'='*100}{RESET}")

    # 1. Obtener Ganadores
    row_pars = df_pars_global[df_pars_global['Dataset'] == dataset]
    if row_pars.empty: continue
    k_master, meth_master = int(row_pars.iloc[0]['K_elegido']), row_pars.iloc[0]['Metodologia']
    k_wl, meth_wl = calculate_win_loss_winner(dataset)

    # 2. Regla de Seguridad
    if k_wl == k_master and meth_wl == meth_master:
        det_path = f"../best_k_selection/parsimony_analysis/metrics_per_k_{dataset}.csv"
        df_det = pd.read_csv(det_path)
        pars_list = df_det[df_det['Elegido'] == 'SI'].sort_values(by='K')
        if len(pars_list) > 1:
            k_contrast, meth_contrast = int(pars_list.iloc[1]['K']), pars_list.iloc[1]['Metodologia']
            motivo = "2º Mejor Parsimonia"
        else:
            k_contrast = 25 if k_master != 25 else 50
            meth_contrast = get_winner_info(dataset, k_contrast)
            motivo = "Modelo alternativo"
    else:
        k_contrast, meth_contrast = k_wl, meth_wl
        motivo = "Ganador Win-Loss Ranking"

    print(f" {BOLD}Configuración Seleccionada:{RESET}")
    print(f"   {BLUE} Maestro (Parsimonia): K={k_master} | {meth_master}{RESET}")
    print(f"   {YELLOW} Contraste ({motivo}): K={k_contrast} | {meth_contrast}{RESET}\n")

    # 3. Carga y Análisis ANOVA
    path_m, path_c = get_dataset_path(dataset, k_master, meth_master), get_dataset_path(dataset, k_contrast, meth_contrast)
    if not os.path.exists(path_m) or not os.path.exists(path_c): continue

    df_m = pd.read_csv(path_m)
    X_m = df_m.drop(columns=['target'])
    f_val, _ = f_classif(X_m, df_m['target'])
    df_rank_m = pd.DataFrame({'Gene': X_m.columns, 'Score': f_val}).sort_values(by='Score', ascending=False).reset_index(drop=True)
    df_rank_m.index += 1

    df_c_cols = pd.read_csv(path_c, nrows=0)
    genes_c = [c for c in df_c_cols.columns if c.lower() != 'target']

    # 4. Clasificación de Genes
    # Mantenemos todos los genes del contraste y buscamos su posición en el maestro
    data_all_contrast = []
    for gene in genes_c:
        match_master = df_rank_m[df_rank_m['Gene'] == gene]
        if not match_master.empty:
            rank_m = match_master.index[0]
            score_m = match_master.iloc[0]['Score']
        else:
            rank_m = "N/A"
            score_m = 0.0
        data_all_contrast.append({'Gene': gene, 'Rank_Maestro': rank_m, 'Score_ANOVA': score_m})
    
    df_contrast_final = pd.DataFrame(data_all_contrast)
    
    # Genes en Maestro pero NO en Contraste
    df_outside = df_rank_m[~df_rank_m['Gene'].isin(genes_c)].copy()

    # 5. Guardado de Archivos
    # TXT: Solo listado de genes del contraste
    txt_path = f"k_value_comparison/genes_{dataset}_k{k_contrast}.txt"
    with open(txt_path, "w", encoding="utf-8") as f: 
        f.write("\n".join(genes_c))
    
    # CSV: Listado completo del contraste con su relevancia en el maestro
    csv_path = f"k_value_comparison/contrast_mapping_{dataset}_k{k_master}-vs-k{k_contrast}.csv"
    df_contrast_final.to_csv(csv_path, index=False)

    # 6. IMPRESIÓN: LISTADO COMPLETO DEL CONTRASTE
    print(f"{BOLD} LISTADO DEL CONTRASTE (Posicionamiento en Maestro K={k_master}){RESET}")
    print(f"{'#':<4} | {'GEN':<20} | {'RANK MAESTRO':<15} | {'SCORE ANOVA':<12}")
    print("-" * 65)
    
    # Ordenamos la impresión para facilitar la lectura
    df_print = df_contrast_final.sort_values(by='Score_ANOVA', ascending=False)
    
    for i, row in enumerate(df_print.itertuples(), 1):
        r_str = f"#{row.Rank_Maestro}" if row.Rank_Maestro != "N/A" else "N/A"
        print(f"{i:<4} | {row.Gene:<20} | {r_str:<15} | {row.Score_ANOVA:<12.4f}")

    # 7. IMPRESIÓN: TOP/BOTTOM FUERA DE INTERSECCIÓN
    print(f"\n{BOLD} GENES FUERA DEL CONTRASTE (Solo en Maestro K={k_master}){RESET}")
    if not df_outside.empty:
        top_out = df_outside.head(3)
        bot_out = df_outside.tail(3)
        print(f"   {GREEN}↑ Top 3 mejores fuera:{RESET}")
        for row in top_out.itertuples():
            print(f"     - {row.Gene:<18} (Rank #{row.Index}, Score: {row.Score:.4f})")
        print(f"   {RED}↓ Bottom 3 peores fuera:{RESET}")
        for row in bot_out.itertuples():
            print(f"     - {row.Gene:<18} (Rank #{row.Index}, Score: {row.Score:.4f})")

    # 8. ESTADÍSTICAS DE SCORE ANOVA
    print(f"\n{BOLD} ESTADÍSTICAS DE RELEVANCIA (Score ANOVA){RESET}")
    
    def get_stats(df_subset, col='Score'):
        if df_subset.empty: return "N/A"
        s = df_subset[col]
        return f"Min: {s.min():.3f} | Max: {s.max():.3f} | Mean: {s.mean():.3f} | Std: {s.std():.3f}"

    print(f"   - {BLUE}Contraste:{RESET}    {get_stats(df_contrast_final, 'Score_ANOVA')}")
    print(f"   - {BLUE}General (K={k_master}):{RESET} {get_stats(df_rank_m)}")
    print(f"   - {BLUE}Fuera Selección:{RESET} {get_stats(df_outside)}")

    print(f"\n {BOLD}Archivos:{RESET} TXT ({txt_path}) | CSV ({csv_path})")

print(f"\n{BOLD} PROCESO FINALIZADO. Reporte guardado en: {log_report_path}{RESET}")


 COMPARATIVA DINÁMICA: BRAAKSC
 Configuración Seleccionada:
    Maestro (Parsimonia): K=50 | BorderlineSMOTE_RF-R2_FR
    Contraste (Ganador Win-Loss Ranking): K=75 | ROS_RF-R2_FR

 LISTADO DEL CONTRASTE (Posicionamiento en Maestro K=50)
#    | GEN                  | RANK MAESTRO    | SCORE ANOVA 
-----------------------------------------------------------------
1    | CIC                  | #1              | 7.2380      
2    | CMTM4                | #2              | 6.9389      
3    | LRRC4B               | #3              | 6.7979      
4    | TMEM107              | #4              | 6.5481      
5    | TMCC2                | #5              | 6.3151      
6    | NIT1                 | #6              | 6.2429      
7    | IP6K2                | #7              | 6.2249      
8    | ZNF592               | #8              | 6.0889      
9    | ADAMTS2              | #9              | 6.0133      
10   | ZNF77                | #10             | 6.0012      
11   | PPDPF            